# 03 — AST (DeiT-Small) Staged Fine-Tuning

The AST branch uses **`timm` DeiT-Small** (`deit_small_patch16_224`, embed_dim=384)
pretrained on ImageNet-21k, adapted for binary audio classification.

## Architecture
- Input: log-mel `[B, 1, 64, 1000]` → repeat to 3 ch → bilinear resize to 224×224
- DeiT-Small backbone (CLS-token pooling, 384-dim)
- Head: `LayerNorm(384)` → `Linear(384, 256)` → `GELU` → `Dropout(0.3)` → `Linear(256, 1)`
- ~22 M parameters total; ~2.4 M in head

## Staged training
| Stage | Frozen | Trainable | LR |
|---|---|---|---|
| 1 (8 ep)  | DeiT backbone | head only | 1e-4 |
| 2 (≤42 ep) | DeiT blocks 0–7 | head + last 4 blocks | 1e-4 |

In [ ]:
import sys
sys.path.append('..')

from src.models import ASTBinaryClassifier

# Build DeiT-Small AST (pretrained=True downloads ImageNet weights via timm)
model = ASTBinaryClassifier(pretrained=True, dropout=0.3)
print(f'Total parameters      : {model.total_parameters()/1e6:.1f} M')
print(f'DeiT embed_dim        : {model.backbone.num_features}')

In [ ]:
# Stage 1 — freeze backbone, only head trainable
model.freeze_backbone()
trainable_s1 = model.trainable_parameters()
print(f'Stage 1 trainable: {trainable_s1 / 1e6:.2f} M  (head only)')

In [ ]:
# Stage 2 — unfreeze last 4 DeiT blocks + head
model.unfreeze_last_blocks(4)
trainable_s2 = model.trainable_parameters()
print(f'Stage 2 trainable: {trainable_s2 / 1e6:.2f} M  (head + last 4 blocks)')
print(f'Number of DeiT blocks total: {len(list(model.backbone.blocks))}')

In [ ]:
import torch

# Forward-pass smoke test: [B=2, 1-ch, 64-mel, 1000-frames]
dummy = torch.randn(2, 1, 64, 1000)
logits = model(dummy)
print(f'Input shape  : {tuple(dummy.shape)}')
print(f'Output shape : {tuple(logits.shape)}')  # [2]
print(f'Logits       : {logits.detach()}')